<a href="https://colab.research.google.com/github/ZataraHere/Basic-Neural-Networks/blob/main/Deep_Neural_Network_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import sklearn
import sklearn.datasets
from sklearn.metrics import accuracy_score


In [8]:
class DeepNeuralNetwork:
  def __init__(self,layer_dims, learning_rate=0.01):
    self.layer_dims = layer_dims
    self.learning_rate = learning_rate
    self.parameters = {}
    self.initialize_parameters()


  def imitialize_paramameters(self):
        np.random.seed(42)
        L = len(self.layer_dims)

        for l in range(1, L):
            # He initialization helps prevent vanishing/exploding gradients with ReLU
            self.parameters[f"W{l}"] = np.random.randn(self.layer_dims[l], self.layer_dims[l-1]) * np.sqrt(2.0 / self.layer_dims[l-1])
            self.parameters[f"b{l}"] = np.zeros((self.layer_dims[l], 1))

  def sigmoid(self, Z):
    return 1/(1+ np.exp(-Z))

  def relu(self,Z):
    return np.maximum(0,Z)



  def forward_propagation(self, X):



In [ ]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

class DeepNeuralNetwork:
    def __init__(self, layer_dims, learning_rate=0.01):
        """
        layer_dims: List containing the dimensions of each layer (e.g., [2, 4, 3, 1])
                    where index 0 is the input size, and the last is the output size.
        """
        self.layer_dims = layer_dims
        self.learning_rate = learning_rate
        self.parameters = {}
        self.initialize_parameters()

    def initialize_parameters(self):
        """Initialize weights with He initialization and biases with zeros."""
        np.random.seed(42)
        L = len(self.layer_dims)

        for l in range(1, L):
            # He initialization helps prevent vanishing/exploding gradients with ReLU
            self.parameters[f"W{l}"] = np.random.randn(self.layer_dims[l], self.layer_dims[l-1]) * np.sqrt(2.0 / self.layer_dims[l-1])
            self.parameters[f"b{l}"] = np.zeros((self.layer_dims[l], 1))

    def _relu(self, Z):
        return np.maximum(0, Z), Z

    def _sigmoid(self, Z):
        # Clip Z to prevent overflow errors in exp
        Z = np.clip(Z, -500, 500)
        A = 1 / (1 + np.exp(-Z))
        return A, Z

    def _relu_backward(self, dA, cache):
        Z = cache
        dZ = np.array(dA, copy=True)
        dZ[Z <= 0] = 0
        return dZ

    def _sigmoid_backward(self, dA, cache):
        Z = cache
        s = 1 / (1 + np.exp(-Z))
        dZ = dA * s * (1 - s)
        return dZ

    def forward_propagation(self, X):
        """Execute forward pass through all layers."""
        caches = []
        A = X
        L = len(self.layer_dims) - 1

        # Hidden layers (ReLU)
        for l in range(1, L):
            A_prev = A
            W = self.parameters[f"W{l}"]
            b = self.parameters[f"b{l}"]

            Z = np.dot(W, A_prev) + b
            A, activation_cache = self._relu(Z)
            caches.append((A_prev, W, b, activation_cache))

        # Output layer (Sigmoid for binary classification)
        W_last = self.parameters[f"W{L}"]
        b_last = self.parameters[f"b{L}"]
        Z_last = np.dot(W_last, A) + b_last
        AL, activation_cache = self._sigmoid(Z_last)
        caches.append((A, W_last, b_last, activation_cache))

        return AL, caches

    def compute_cost(self, AL, Y):
        """Compute Binary Cross-Entropy loss."""
        m = Y.shape[1]
        # Clip AL to avoid log(0)
        AL = np.clip(AL, 1e-15, 1 - 1e-15)
        cost = -1 / m * np.sum(Y * np.log(AL) + (1 - Y) * np.log(1 - AL))
        return np.squeeze(cost)

    def backward_propagation(self, AL, Y, caches):
        """Execute backward pass to calculate gradients."""
        grads = {}
        L = len(caches)
        m = AL.shape[1]
        Y = Y.reshape(AL.shape)

        # Derivative of cost with respect to AL
        dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL))

        # Output layer backward (Sigmoid)
        current_cache = caches[L-1]
        A_prev, W, b, activation_cache = current_cache
        dZ = self._sigmoid_backward(dAL, activation_cache)
        grads[f"dW{L}"] = 1 / m * np.dot(dZ, A_prev.T)
        grads[f"db{L}"] = 1 / m * np.sum(dZ, axis=1, keepdims=True)
        dA_prev = np.dot(W.T, dZ)

        # Hidden layers backward (ReLU)
        for l in reversed(range(L-1)):
            current_cache = caches[l]
            A_prev, W, b, activation_cache = current_cache
            dZ = self._relu_backward(dA_prev, activation_cache)
            grads[f"dW{l+1}"] = 1 / m * np.dot(dZ, A_prev.T)
            grads[f"db{l+1}"] = 1 / m * np.sum(dZ, axis=1, keepdims=True)
            dA_prev = np.dot(W.T, dZ)

        return grads

    def update_parameters(self, grads):
        """Update weights and biases using Gradient Descent."""
        L = len(self.layer_dims) - 1
        for l in range(1, L + 1):
            self.parameters[f"W{l}"] -= self.learning_rate * grads[f"dW{l}"]
            self.parameters[f"b{l}"] -= self.learning_rate * grads[f"db{l}"]

    def fit(self, X, Y, epochs=1000, print_cost=True):
        """Train the neural network."""
        for i in range(epochs):
            AL, caches = self.forward_propagation(X)
            cost = self.compute_cost(AL, Y)
            grads = self.backward_propagation(AL, Y, caches)
            self.update_parameters(grads)

            if print_cost and i % (epochs // 10 or 1) == 0:
                print(f"Cost after epoch {i}: {cost:.4f}")

    def predict(self, X):
        """Make predictions (0 or 1)."""
        AL, _ = self.forward_propagation(X)
        return (AL > 0.5).astype(int)